In [1]:
import torch
import torch.nn as nn
import math

In [3]:
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank=4, alpha=8):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.lora_A = nn.Parameter(torch.randn(rank, in_dim))
        self.lora_B = nn.Parameter(torch.zeros(out_dim, rank))
        self.scaling = alpha / rank
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        return (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

In [4]:
class LinearWithLoRA(nn.Module):
    def __init__(self, original_linear, rank=4, alpha=8):
        super().__init__()
        self.original_linear = original_linear

        for param in self.original_linear.parameters():
            param.requires_grad = False

        self.lora_adapter = LoRALayer(
            original_linear.in_features,
            original_linear.out_features,
            rank,
            alpha
        )

    def forward(self, x):
        return self.original_linear(x) + self.lora_adapter(x)

In [5]:
frozen_layer = nn.Linear(10, 5)
x = torch.randn(1, 10)
original_output = frozen_layer(x)

In [6]:
lora_model = LinearWithLoRA(frozen_layer, rank=2)
lora_output = lora_model(x)

In [7]:
print(f"Original: {original_output}")
print(f"LoRA    : {lora_output}")
print(f"Match?  : {torch.allclose(original_output, lora_output)}")

Original: tensor([[ 0.5747, -1.0012, -0.4544, -0.2457, -0.0885]],
       grad_fn=<AddmmBackward0>)
LoRA    : tensor([[ 0.5747, -1.0012, -0.4544, -0.2457, -0.0885]], grad_fn=<AddBackward0>)
Match?  : True


In [8]:
total_params = sum(p.numel() for p in lora_model.parameters())
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

In [9]:
print(f"\nTotal Params: {total_params}")
print(f"Trainable:    {trainable_params} (Only {trainable_params/total_params:.1%}!)")


Total Params: 85
Trainable:    30 (Only 35.3%!)
